# Modèle de prédiction du risque de crise cardiaque

Notebook d'entraînement — équipe Data Science.

Objectif : à partir d'un profil patient (âge, tension, cholestérol, résultats ECG, etc.), prédire la présence d'un risque de crise cardiaque.

**Statut : exploration terminée, modèle validé. Rien n'est packagé ni déployé — c'est l'objet du TP Module 5.**

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import joblib

df = pd.read_csv("../data/raw/heart_train.csv")
df.head()

## 1. Description des données

| Colonne | Description |
|---|---|
| age | Âge du patient |
| sex | Sexe (1 = homme, 0 = femme) |
| cp | Type de douleur thoracique (0-3) |
| trestbps | Tension artérielle au repos (mm Hg) |
| chol | Cholestérol sérique (mg/dl) |
| fbs | Glycémie à jeun > 120 mg/dl (1 = oui) |
| restecg | Résultat ECG au repos (0-2) |
| thalach | Fréquence cardiaque maximale atteinte |
| exang | Angine induite par l'effort (1 = oui) |
| oldpeak | Dépression du segment ST induite par l'effort |
| slope | Pente du segment ST à l'effort (0-2) |
| ca | Nombre de vaisseaux principaux colorés (0-3) |
| thal | Thalassémie (1 = normal, 2 = défaut fixe, 3 = défaut réversible) |
| target | 1 = risque de crise cardiaque présent, 0 = absent |

In [ ]:
df.describe()

In [ ]:
df["target"].value_counts(normalize=True)

Classes globalement équilibrées (~46% de cas positifs). Pas de valeurs manquantes sur ce jeu.

## 2. Séparation train / test

In [ ]:
FEATURES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal",
]

X = df[FEATURES]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

## 3. Entraînement

RandomForest choisi après quelques essais rapides (régression logistique et RandomForest) : meilleur rappel sur la classe positive, ce qui est le critère prioritaire pour un usage médical (un faux négatif est plus coûteux qu'un faux positif).

In [ ]:
model = RandomForestClassifier(
    n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=42
)
model.fit(X_train, y_train)

## 4. Évaluation

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("recall:  ", round(recall_score(y_test, y_pred), 4))
print("f1:      ", round(f1_score(y_test, y_pred), 4))
print("auc:     ", round(roc_auc_score(y_test, y_proba), 4))
print("confusion matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
importances

Les variables les plus discriminantes sont cohérentes avec la littérature clinique : `cp` (type de douleur thoracique), `thal`, `ca`, `oldpeak`, `exang`.

## 5. Sauvegarde du modèle

Le modèle validé est sauvegardé en `.pkl`. **À partir d'ici, ce n'est plus le travail de la data scientist : le packaging, le déploiement, le versionnement et le monitoring sont à la charge de l'équipe MLOps (TP Module 5).**

In [ ]:
joblib.dump({"model": model, "features": FEATURES}, "../heart_risk_model.pkl")